In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch01-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "01-01-molecular-graphs.py"

# 乙醇和二甲醚的氧各連著誰？

改動下一格的 Python 腳本後，請重新執行該格與最後一格。ESOL 首次載入需要下載，這裡只讀取結構。

<!-- concept-ch00-05:start -->
沿著氧周圍的鍵逐一讀，就能區分下面三個結構。

| 分子 | 氧的碳鄰居與鍵 | 氧上的氫 |
|---|---|---|
| 乙醇 CH₃CH₂OH | 一個碳，以單鍵相連 | 一個 |
| 二甲醚 CH₃OCH₃ | 兩個碳，各以單鍵相連 | 沒有 |
| 乙醛 CH₃CHO | 一個碳，以雙鍵相連 | 沒有 |

乙醇與乙醛的氧都只有一個碳鄰居，但鍵級與氫數不同；分子圖必須同時保存這些資訊。

<!-- concept-ch00-05:end -->

## 結構式與隱含氫

有機化學常用的骨架式會省略碳的元素符號，也通常不畫出接在碳上的氫，讓碳骨架和官能基更容易看清楚。讀圖時，再依成鍵情形補回氫數。例如乙醇末端的碳只畫出一條碳碳單鍵，還要接三個氫；中間的碳已分別接到碳和氧，所以再接兩個氫。氧上的氫通常會寫出來，標成 OH，方便讀者認出羥基。省略的是符號，分子裡的氫並沒有少掉。

這個判讀方法以原子的成鍵規則為依據。對本節的中性乙醇和二甲醚，可以用碳的鍵級總和為四、氧為二來核對；單鍵算一，雙鍵算二。乙醇的氫數依序是三、二、一，二甲醚則是三、零、三，兩者加總都是六。

### 電腦為什麼也使用隱含氫？

電腦可以替每個氫建立一筆原子紀錄，也可以先記下碳、氧等原子及其成鍵關係，再按元素、鍵級與電荷等資訊推算附帶的氫數。後一種做法稱為**隱含氫**（implicit hydrogen）。本節讀入乙醇的 `CCO` 時，RDKit 建立的是兩個碳和一個氧的原子紀錄；六個氫沒有各自成為獨立的節點，但程式仍能查到每個原子帶幾個氫。

這樣表示，是為了讓資料和處理步驟更精簡。像辨認醇與醚這類工作，主要需要查碳、氧的連接關係及其氫數，不必每次都逐一走訪六個氫節點。以乙醇為例，若把氫全部列出，原子紀錄會從三筆增加到九筆，鍵的紀錄也會從兩筆增加到八筆；兩種表示描述的仍是同一個分子。使用隱含氫是一種可依工作需要選擇的表示方式，電腦並非只能這樣存。

還要分清楚「圖上有沒有畫出 H」與「資料裡有沒有獨立的氫原子紀錄」。RDKit 可以根據隱含氫數，把乙醇的氧畫成 OH；畫面上看得到 H，不代表資料裡已多建一個氫節點。因此，本節查氧的鄰居時只會列出 C，氧上的氫數則另外查詢。

若要建立含氫的三維座標、進行需要逐一處理氫原子的計算，或指定氘等氫同位素，就需要適當保留氫原子的明確資訊。帶電原子、自由基等情況，也不能只套用「碳四、氧二」來補氫；讀入的格式和原子的化學狀態都必須一併考慮。



In [ ]:
%%writefile "{script}"
# 概念示意 — 實際應用請使用下方 DeepChem 版本
from pathlib import Path
import os
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

def oxygen_neighbors(mol):
    oxygen = next(atom for atom in mol.GetAtoms() if atom.GetSymbol() == "O")
    neighbors = [atom.GetSymbol() for atom in oxygen.GetNeighbors()]
    return oxygen.GetIdx(), neighbors, oxygen.GetTotalNumHs()

examples = [("Ethanol", "CCO"), ("Dimethyl ether", "COC")]
mols = [Chem.MolFromSmiles(smiles) for _, smiles in examples]
for (name, _), mol in zip(examples, mols):
    print(name, rdMolDescriptors.CalcMolFormula(mol), oxygen_neighbors(mol))
    for bond in mol.GetBonds():
        print(" ", bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond.GetBondType())

options = Draw.MolDrawOptions()
options.addAtomIndices = True
options.explicitMethyl = True
options.legendFontSize = 24
panel = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(360, 230),
                             legends=[name for name, _ in examples], drawOptions=options)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
panel.save(out / "01-01-molecular-graphs.png")

# DeepChem 版本：保留 ESOL 資料列，逐列算分子式，再讀取氧的鄰居。
import deepchem as dc

_, parts, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="index",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
for part in parts:
    for smiles in part.ids:
        mol = Chem.MolFromSmiles(smiles)
        if rdMolDescriptors.CalcMolFormula(mol) == "C2H6O":
            print("ESOL", smiles, oxygen_neighbors(mol))

In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "01-01-molecular-graphs.png")))